# 2 — Results Analysis

This notebook is the **offline analysis companion** to `1_run_attribution.ipynb`.
It requires no GPU, no model loading, and no Runpod connection.
It loads a completed run directory from `my_work/results/runs/` and produces
thesis-ready summary tables, charts, and observations.

**Prerequisites:**
1. `1_run_attribution.ipynb` has completed on Runpod.
2. The run directory has been committed and pushed to GitHub.
3. You have pulled the latest changes locally (`git pull`).

Set `RUN_ID` in the Configuration cell below to the run you want to analyse.

---

## Imports

Standard analysis stack only — no GPU dependencies.

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print('Imports OK')
print(f'pandas  {pd.__version__}')
print(f'matplotlib  {plt.matplotlib.__version__}')

---

## Configuration

Set `RUN_ID` to the run you want to analyse. Leave it as `None` to automatically select the
most recent run in the results directory.

In [ ]:
# Set to a specific run_id string, or None to auto-select the most recent run.
RUN_ID: str | None = None

def _find_results_root() -> Path:
    """Locate my_work/results/runs/ from CWD or common workspace paths."""
    cwd = Path.cwd().resolve()
    candidates = []
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / 'my_work' / 'results' / 'runs')
    for root in [Path('/workspace'), Path('/root')]:
        if root.is_dir():
            try:
                for child in root.iterdir():
                    if child.is_dir():
                        candidates.append(child / 'my_work' / 'results' / 'runs')
            except Exception:
                pass
    candidates.append(
        Path('/Users/Lacus/Documents/ELTE/Thesis/circuit-tracer-main/my_work/results/runs')
    )
    for c in candidates:
        if c.is_dir():
            return c
    raise FileNotFoundError(
        f'Could not locate my_work/results/runs/. CWD={cwd}. '
        'Run 1_run_attribution.ipynb first and pull its outputs.'
    )


RUNS_ROOT = _find_results_root()
print(f'Runs root: {RUNS_ROOT}')

available_runs = sorted([d.name for d in RUNS_ROOT.iterdir() if d.is_dir()])
print(f'Available runs ({len(available_runs)}):')
for r in available_runs:
    print(f'  {r}')

if RUN_ID is None:
    RUN_ID = available_runs[-1]
    print(f'\nAuto-selected most recent run: {RUN_ID}')
else:
    assert RUN_ID in available_runs, f'Run not found: {RUN_ID}'
    print(f'\nSelected run: {RUN_ID}')

RUN_DIR = RUNS_ROOT / RUN_ID

---

## Load Run Artifacts

All tables and metadata are loaded from the run directory. No model is needed.

In [ ]:
manifest   = json.loads((RUN_DIR / 'manifest.json').read_text())
snapshot   = json.loads((RUN_DIR / 'questions_snapshot.json').read_text())
summary    = json.loads((RUN_DIR / 'tables' / 'summary_metrics.json').read_text())

df_baseline  = pd.read_csv(RUN_DIR / 'tables' / 'baseline_results.csv')
df_features  = pd.read_csv(RUN_DIR / 'tables' / 'top_features.csv')
df_frequency = pd.read_csv(RUN_DIR / 'tables' / 'feature_frequency.csv')
df_interv    = pd.read_csv(RUN_DIR / 'tables' / 'interventions.csv')

QUESTIONS       = snapshot['questions']
TEMPLATES       = snapshot['templates']
ACTIVE_TEMPLATE = snapshot['metadata']['active_template']

print(f"Run ID          : {manifest['run_id']}")
print(f"Timestamp (UTC) : {manifest['timestamp_utc']}")
print(f"Git SHA         : {manifest['git_sha']}")
print(f"Model           : {manifest['model_name']}")
print(f"Transcoder      : {manifest['transcoder_name']}")
print(f"Backend         : {manifest['backend']}")
print(f"Active template : {ACTIVE_TEMPLATE}")
print(f"Questions       : {manifest['n_questions']}")
print(f"Baseline rows   : {len(df_baseline)}")
print(f"Top-feature rows: {len(df_features)}")
print(f"Frequency rows  : {len(df_frequency)}")
print(f"Intervention rows: {len(df_interv)}")

---

## Section 1 — Baseline Analysis

We examine the model's unaided accuracy and prediction margins across all questions and templates.

A positive margin means the model predicted `True`. The magnitude indicates confidence.
Questions where the model is *wrong with high confidence* (large margin, wrong label) are the
most interesting cases for circuit analysis because the circuit is clearly executing but producing
the incorrect answer.

In [ ]:
print('=== BASELINE ACCURACY BY TEMPLATE ===')
for tkey, stats in summary['accuracy_by_template'].items():
    print(f"  {tkey}: {stats['correct']}/{stats['total']} ({stats['pct']:.1f}%)")

print()
print(f'=== BASELINE MARGINS — TEMPLATE {ACTIVE_TEMPLATE} ===')
df_active = df_baseline[df_baseline['template'] == ACTIVE_TEMPLATE].copy()
df_active['sign'] = df_active['margin'].apply(lambda m: '+' if m > 0 else '-')
df_active['ok']   = df_active['correct'].apply(lambda c: 'OK' if c else 'WRONG')

print(df_active[['id', 'label', 'logit_true', 'logit_false', 'margin', 'ok']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: margin by question ──────────────────────────────────────────────────
ax = axes[0]
colors = ['steelblue' if c else 'tomato' for c in df_active['correct']]
ax.barh(df_active['id'], df_active['margin'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('True-False logit margin')
ax.set_title(f'Baseline Margins (template {ACTIVE_TEMPLATE})')
ax.invert_yaxis()
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color='steelblue', label='Correct'),
    plt.Rectangle((0, 0), 1, 1, color='tomato',    label='Wrong'),
]
ax.legend(handles=legend_handles, loc='lower right')

# ── Right: accuracy by template ───────────────────────────────────────────────
ax2  = axes[1]
tkeys = list(summary['accuracy_by_template'].keys())
pcts  = [summary['accuracy_by_template'][t]['pct'] for t in tkeys]
ax2.bar(tkeys, pcts, color='mediumpurple')
ax2.set_ylim(0, 105)
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy by Template')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.axhline(50, color='gray', linestyle='--', linewidth=0.8, label='chance')
ax2.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / 'tables' / 'baseline_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to', RUN_DIR / 'tables' / 'baseline_overview.png')

---

## Section 2 — Top Feature Analysis

For each question, the runner notebook saved the top `TOP_N` features ranked by multi-hop influence.
Here we examine:

- Which features appear at the top for each question.
- Whether specific layers are over-represented.
- How influence score magnitudes compare across questions (using the normalised `score_norm` column).

The `score_norm` column rescales each question's scores to [-1, 1] so that questions with
small probability gaps (low `prob` weight) remain comparable to high-confidence questions.

In [ ]:
print(f'Top features table: {len(df_features)} rows across {df_features["qid"].nunique()} questions')
print()
print('Top 5 features by score_norm for each question:')
print()
for qid in sorted(df_features['qid'].unique()):
    sub = df_features[df_features['qid'] == qid].nlargest(5, 'score_norm')
    label = next(q['label'] for q in QUESTIONS if q['id'] == qid)
    print(f"  {qid} (label={'True' if label else 'False'}):")
    for _, row in sub.iterrows():
        print(f"    rank={int(row['rank']):2}  ({int(row['layer'])}, {int(row['position'])}, {int(row['feature_idx']):<8})  score_norm={row['score_norm']:+.4f}  {row['sign']}")
    print()

In [ ]:
# Layer distribution of top features
layer_counts = df_features.groupby('layer').size().reset_index(name='count')
layer_counts = layer_counts.sort_values('layer')

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(layer_counts['layer'], layer_counts['count'], color='teal')
ax.set_xlabel('Layer')
ax.set_ylabel('Count in top-feature lists')
ax.set_title('Layer Distribution of Top Features')
plt.tight_layout()
plt.savefig(RUN_DIR / 'tables' / 'layer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

---

## Section 3 — Cross-Question Feature Recurrence

Features that recur across many questions are candidates for shared circuit components —
internal features the model uses broadly for this class of reasoning rather than for any
single prompt.

The frequency table records how many questions each feature appeared in and which ones.

In [ ]:
df_freq_sorted = df_frequency.sort_values('frequency', ascending=False)

print('Features appearing in 2+ questions:')
print()
top_recurring = df_freq_sorted[df_freq_sorted['frequency'] >= 2]
print(top_recurring[['layer', 'position', 'feature_idx', 'frequency', 'question_ids']].to_string(index=False))
print()
print(f'Total unique features: {len(df_frequency)}')
print(f'Recurring (freq >= 2): {len(top_recurring)}')
print(f'Recurring (freq >= 3): {len(df_freq_sorted[df_freq_sorted["frequency"] >= 3])}')

In [ ]:
# Recurrence frequency histogram
fig, ax = plt.subplots(figsize=(7, 3))
max_freq = int(df_frequency['frequency'].max())
ax.hist(df_frequency['frequency'], bins=range(1, max_freq + 2), color='darkorange', rwidth=0.8)
ax.set_xlabel('Number of questions feature appeared in')
ax.set_ylabel('Number of features')
ax.set_title('Feature Recurrence Distribution')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig(RUN_DIR / 'tables' / 'recurrence_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

---

## Section 4 — Intervention Analysis

Circuit interventions test whether the recurring features are causally responsible for the
model's True/False decision. We examine:

- **Amplify (5x)**: did the True-False margin increase (for questions where True features were amplified)?
- **Ablate (0x)**: did the margin decrease or the prediction flip?

A consistent directional effect across multiple questions supports the causal interpretation.

In [ ]:
print('=== INTERVENTION SUMMARY ===')
for itype, stats in summary['intervention_summary'].items():
    print(f"  {itype:<14}: avg delta = {stats['avg_delta_margin']:+.4f}  "
          f"flipped: {stats['n_flipped']}/{stats['total']}")

print()
print('=== PER-QUESTION DETAILS ===')
print(df_interv[['id', 'label', 'intervention', 'n_features', 'margin_before',
                  'margin_after', 'delta_margin', 'flipped']].to_string(index=False))

In [ ]:
# Delta margin by question and intervention type
fig, ax = plt.subplots(figsize=(11, 4))

qids   = sorted(df_interv['id'].unique())
x      = range(len(qids))
width  = 0.35

for i, (itype, color) in enumerate([('amplify_5x', 'steelblue'), ('ablate', 'tomato')]):
    subset = df_interv[df_interv['intervention'] == itype].set_index('id')
    deltas = [subset.loc[q, 'delta_margin'] if q in subset.index else 0.0 for q in qids]
    offset = (i - 0.5) * width
    ax.bar([xi + offset for xi in x], deltas, width, label=itype, color=color, alpha=0.8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(qids, rotation=30, ha='right')
ax.set_ylabel('Delta margin (after - before)')
ax.set_title('Intervention Effect on True-False Margin')
ax.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / 'tables' / 'intervention_deltas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

---

## Section 5 — Key Observations

This section is a space for you to record your interpretation of the results.
The prompts below suggest useful comparisons. Replace or extend them as you develop insights.

Questions to address:

1. **Accuracy**: Does the model distinguish True from False in this domain? Is there a template effect?
2. **Top features**: Are there features that consistently appear in the top-5 across most questions? What layers are they in?
3. **Recurrence**: How many features recur across 3 or more questions? Is recurrence stronger for True-labelled or False-labelled questions?
4. **Interventions**: Did amplification consistently increase the True-False margin? Did ablation reduce or flip predictions? Is the effect stronger for True or False items?
5. **Interpretation**: Based on Neuronpedia links in the attribution table, can any recurring features be labelled semantically?

In [ ]:
# Workspace — add your analysis and notes here.


---

## Appendix — Run Metadata

In [ ]:
print(json.dumps(manifest, indent=2))